# Get code working

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [2]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

# Animate Normalised Demand
- Goal: Animate normalised demand in Greater Sydney Region on each national public holiday
- Want to add geographical lines to better show the region
- Adapting code I have already written to include geographical lines

## Adding lat and lon to dataframe

In [8]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, Sydney, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

## Holiday Dictionary 

In [9]:
from dateutil.easter import easter

# -------------------------------
# Holiday functions and registry
# -------------------------------

def new_years_day(year): return pd.Timestamp(f"{year}-01-01")
def australia_day(year): return pd.Timestamp(f"{year}-01-26")
def anzac_day(year): return pd.Timestamp(f"{year}-04-25")
def good_friday(year): return easter(year) - timedelta(days=2)
def easter_saturday(year): return easter(year) - timedelta(days=1)
def easter_sunday(year): return easter(year)
def easter_monday(year): return easter(year) + timedelta(days=1)
def christmas_day(year): return pd.Timestamp(f"{year}-12-25")
def boxing_day(year): return pd.Timestamp(f"{year}-12-26")

HOLIDAYS = {
    "New Year’s Day": new_years_day,
    "Australia Day": australia_day,
    "ANZAC Day": anzac_day,
    "Good Friday": good_friday,
    "Easter Saturday": easter_saturday,
    "Easter Sunday": easter_sunday,
    "Easter Monday": easter_monday,
    "Christmas Day": christmas_day,
    "Boxing Day": boxing_day,
}


## 1. Making the animation
- creates the function to plot and animate the demand
- includes geographical lines

In [67]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
import pandas as pd
import os
import numpy as np
import cartopy.crs as ccrs

def animate_sydney_heatmap(demand, info, date,
                           lat_col="latitude", lon_col="longitude",
                           cmap="viridis", fps=4,
                           city_locations=None,
                           city_marker_style=None,
                           output_dir="/home/565/pv3484/aus_substation_electricity/figures/map_animation/Map_Lines_Added",
                           filename_prefix="demand"):
    """
    Animated heatmap of electricity demand over 24 hours for a given day.
    Uses lat/lon from `info`. Demand is normalized per substation (0–1).
    Adds coastlines for geographic context.
    Overlays city markers with distinct styles and a stable legend.
    Layout uses GridSpec to keep map, colorbar, and legend aligned.
    """

    target_date = pd.to_datetime(date).normalize()
    demand.index = pd.to_datetime(demand.index)

    # Filter for the given day
    day_data = demand[demand.index.normalize() == target_date]
    if day_data.empty:
        raise ValueError(f"No demand data found for {target_date.date()}.")

    # Reshape wide → long
    day_long = day_data.reset_index().melt(
        id_vars=[day_data.index.name or 'index'],
        var_name="substation",
        value_name="demand"
    )
    day_long.rename(columns={day_data.index.name or 'index': "timestamp"}, inplace=True)

    # Reset info index safely
    info_reset = info.reset_index().rename(columns={info.index.name or "index": "substation"})
    info_reset = info_reset.loc[:, ~info_reset.columns.duplicated()]

    # Merge demand + metadata
    merged = day_long.merge(info_reset, on="substation", how="left")
    if merged.empty:
        raise ValueError("Merge produced no rows. Check substation IDs and info index alignment.")

    # Bounding box of Sydney data
    lon_min, lon_max = merged[lon_col].min(), merged[lon_col].max()
    lat_min, lat_max = merged[lat_col].min(), merged[lat_col].max()

    # Normalize demand per substation
    merged["norm_demand"] = merged.groupby("substation")["demand"].transform(
        lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
    )

    # Unique timestamps
    timestamps = merged['timestamp'].sort_values().unique()

    # --- Layout with GridSpec ---
    fig = plt.figure(figsize=(12,6))
    gs = gridspec.GridSpec(1, 3, width_ratios=[6, 0.3, 1], figure=fig)

    ax_main = fig.add_subplot(gs[0], projection=ccrs.PlateCarree())
    cbar_ax = fig.add_subplot(gs[1])   # dedicated colorbar axis
    legend_ax = fig.add_subplot(gs[2]) # dedicated legend axis (blank)

    # --- Main Sydney demand map ---
    ax_main.coastlines(resolution="50m", linewidth=1)

    # Auto ticks
    lon_ticks = np.linspace(lon_min, lon_max, 5)
    lat_ticks = np.linspace(lat_min, lat_max, 5)
    ax_main.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
    ax_main.set_yticks(lat_ticks, crs=ccrs.PlateCarree())
    ax_main.set_xlabel("Longitude", fontsize=12)
    ax_main.set_ylabel("Latitude", fontsize=12)

    # Initial scatter
    subset0 = merged[merged['timestamp'] == timestamps[0]]
    sc = ax_main.scatter(subset0[lon_col], subset0[lat_col],
                         c=subset0["norm_demand"], cmap=cmap, s=100,
                         vmin=0, vmax=1,
                         transform=ccrs.PlateCarree())

    # Colorbar in its own axis
    cbar = plt.colorbar(sc, cax=cbar_ax)
    cbar.set_label("Normalized Demand (0–1)")

    ax_main.set_title(f"Sydney Demand (normalized per substation) on {target_date.date()} at {timestamps[0].strftime('%H:%M')}")

    # --- Add city markers + stable legend ---
    handles = []
    if city_locations:
        default_styles = {
            "Sydney CBD": {"color": "red", "marker": "o"},
            "Bondi Junction": {"color": "blue", "marker": "s"},
            "Parramatta": {"color": "green", "marker": "^"},
            "Chatswood": {"color": "teal", "marker": "D"},
            "Cronulla": {"color": "orange", "marker": "v"}
        }
        if city_marker_style:
            for city, style in city_marker_style.items():
                default_styles[city] = {**default_styles.get(city, {}), **style}

        for city, (lat, lon) in city_locations.items():
            style = default_styles.get(city, {"color": "black", "marker": "o"})
            h = ax_main.scatter(lon, lat,
                                color=style["color"],
                                s=80,
                                marker=style["marker"],
                                transform=ccrs.PlateCarree(),
                                zorder=5)
            h.set_label(city)
            handles.append(h)

    # Legend drawn in its own subplot
    legend_ax.axis("off")
    legend_ax.legend(handles=handles,
                     loc="center",
                     title="City Locations",
                     frameon=False)

    def update(frame):
        ts = timestamps[frame]
        subset = merged[merged['timestamp'] == ts]
        sc.set_offsets(subset[[lon_col, lat_col]].values)
        sc.set_array(subset["norm_demand"].values)
        ax_main.set_title(f"Sydney Demand (normalized per substation) on {target_date.date()} at {ts.strftime('%H:%M')}")
        return sc,

    ani = animation.FuncAnimation(fig, update, frames=len(timestamps), blit=False)

    # Save
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{target_date.date()}_normalized.gif")
    print("Saving to:", output_path)
    ani.save(output_path, writer="pillow", fps=fps)
    plt.close(fig)
    return ani

In [68]:
city_locations = {
    "Sydney CBD": (-33.8788, 151.2093),
    "Bondi Junction": (-33.8915, 151.2490),
    "Parramatta": (-33.8150, 151.0011),
    "Chatswood": (-33.7960, 151.1830),
    "Cronulla": (-34.05, 151.15)   # southern location
}

animate_sydney_heatmap(demand, info, "2008-12-25", city_locations=city_locations)

Saving to: /home/565/pv3484/aus_substation_electricity/figures/map_animation/Map_Lines_Added/2008-12-25_normalized.gif


In [29]:
# Example: run animation for Australia Day 2008
animate_sydney_heatmap(
    demand=demand,
    info=info,
    date="2008-01-26",   # Australia Day 2008
    output_dir="./animations",   # folder to save the GIF
    filename_prefix="australia_day"
)

ValueError: No demand data found for 2025-12-25.

## 2. Looping and filing
- uses the first function to loop through all holidays in dictionary
- files them appropriately

In [70]:
from dateutil.easter import easter
from datetime import timedelta
import pandas as pd
import os

# Expand holidays into lists of dates

YEARS = range(2004, 2019)

HOLIDAYS_EXPANDED = {
    name: [func(year) for year in YEARS]
    for name, func in HOLIDAYS.items()
}

# Batch animation loop

base_dir = "/home/565/pv3484/aus_substation_electricity/figures/map_animation/Map_Lines_Added"

for holiday_name, date_list in HOLIDAYS_EXPANDED.items():

    # Create folder for this holiday
    holiday_dir = os.path.join(base_dir, holiday_name)
    os.makedirs(holiday_dir, exist_ok=True)

    print(f"\n=== Processing {holiday_name} ===")

    for date in date_list:
        try:
            print(f"  → {date.date()}")

            animate_sydney_heatmap(
                demand,
                info,
                date,
                city_locations=city_locations,
                city_marker_style=city_marker_style,
                output_dir=holiday_dir,
                filename_prefix=holiday_name
            )

        except Exception as e:
            print(f"  !! Skipping {date.date()} — {e}")



=== Processing New Year’s Day ===
  → 2004-01-01
  !! Skipping 2004-01-01 — name 'city_marker_style' is not defined
  → 2005-01-01
  !! Skipping 2005-01-01 — name 'city_marker_style' is not defined
  → 2006-01-01
  !! Skipping 2006-01-01 — name 'city_marker_style' is not defined
  → 2007-01-01
  !! Skipping 2007-01-01 — name 'city_marker_style' is not defined
  → 2008-01-01
  !! Skipping 2008-01-01 — name 'city_marker_style' is not defined
  → 2009-01-01
  !! Skipping 2009-01-01 — name 'city_marker_style' is not defined
  → 2010-01-01
  !! Skipping 2010-01-01 — name 'city_marker_style' is not defined
  → 2011-01-01
  !! Skipping 2011-01-01 — name 'city_marker_style' is not defined
  → 2012-01-01
  !! Skipping 2012-01-01 — name 'city_marker_style' is not defined
  → 2013-01-01
  !! Skipping 2013-01-01 — name 'city_marker_style' is not defined
  → 2014-01-01
  !! Skipping 2014-01-01 — name 'city_marker_style' is not defined
  → 2015-01-01
  !! Skipping 2015-01-01 — name 'city_marker_sty

AttributeError: 'datetime.date' object has no attribute 'date'